In [1]:
import os

# Configuración de carpetas destino
BASE_DIR = "."  # Directorio actual
REAL_MINI_DIR = os.path.join(BASE_DIR, "Real_mini")
FAKE_MINI_DIR = os.path.join(BASE_DIR, "Fake_mini")

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')
CATEGORIES = ["deepfakes", "face2face", "faceshifter", "faceswap", "neuraltextures"]

registros_recreados = []

# -------------------------------------------------------------
# 1. Reconstruir registros de imágenes 'Real' (Etiqueta 0)
# -------------------------------------------------------------
if os.path.exists(REAL_MINI_DIR):
    # Listar y ordenar para mantener consistencia
    real_files = sorted([f for f in os.listdir(REAL_MINI_DIR) if f.lower().endswith(VALID_EXTENSIONS)])
    
    for filename in real_files:
        file_path = os.path.abspath(os.path.join(REAL_MINI_DIR, filename))
        registros_recreados.append((0, 'real', file_path))

# -------------------------------------------------------------
# 2. Reconstruir registros de imágenes 'Fake' (Etiqueta 1)
# -------------------------------------------------------------
if os.path.exists(FAKE_MINI_DIR):
    fake_files = sorted([f for f in os.listdir(FAKE_MINI_DIR) if f.lower().endswith(VALID_EXTENSIONS)])
    
    for filename in fake_files:
        file_path = os.path.abspath(os.path.join(FAKE_MINI_DIR, filename))
        
        # Identificar qué categoría es según la parte inicial del nombre del archivo (ej. "deepfakes_001.jpg")
        cat_encontrada = 'unknown'
        filename_lower = filename.lower()
        
        for cat in CATEGORIES:
            if filename_lower.startswith(cat):
                cat_encontrada = cat
                break
                
        registros_recreados.append((1, cat_encontrada, file_path))

# -------------------------------------------------------------
# 3. Convertir a Tupla Final
# -------------------------------------------------------------
dataset_tuple = tuple(registros_recreados)

print(f"✓ Tupla recreada con éxito. Total de elementos: {len(dataset_tuple)}")

# Mostrar los primeros 5 ejemplos para verificar la estructura
print("\nPrimeros 5 elementos de la tupla:")
for elem in dataset_tuple[:5]:
    print(elem)

✓ Tupla recreada con éxito. Total de elementos: 1000

Primeros 5 elementos de la tupla:
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_001.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_002.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_003.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_004.jpg')
(0, 'real', 'd:\\OneDrive - UNIR\\Master\\99.TFM\\DataSets\\Real_mini\\real_005.jpg')


In [2]:
import json

# Guardar la tupla en un archivo JSON
with open("dataset_tuple.json", "w", encoding="utf-8") as f:
    json.dump(dataset_tuple, f, ensure_ascii=False, indent=4)

In [ ]:
import requests
import json
import os
import time

# Configuración de credenciales de DeepfakeDetector.ai
API_KEY = ''
BASE_URL = 'https://app.deepfakedetector.ai/api/v1'

filename_json = "respuestas_modelos.json"

# 1. Cargar las rutas e información de las imágenes desde dataset_tuple.json
if os.path.exists("dataset_tuple.json"):
    with open("dataset_tuple.json", "r", encoding="utf-8") as f:
        dataset_tuple = json.load(f)
    print(f"✓ Cargadas {len(dataset_tuple)} imágenes desde 'dataset_tuple.json'")
else:
    print("❌ Error: No se encontró el archivo 'dataset_tuple.json'.")
    dataset_tuple = []

# 2. Cargar o inicializar 'respuestas_modelos'
if os.path.exists(filename_json):
    with open(filename_json, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)
    print(f"✓ Cargadas {len(respuestas_modelos)} respuestas previas desde '{filename_json}'")
else:
    # Inicializar lista con [None, None, None] para cada imagen si el archivo no existe
    respuestas_modelos = [[None, None, None] for _ in range(len(dataset_tuple))]

# Ajustar el tamaño si hay más elementos en el dataset que en respuestas_modelos
if len(respuestas_modelos) < len(dataset_tuple):
    for _ in range(len(dataset_tuple) - len(respuestas_modelos)):
        respuestas_modelos.append([None, None, None])

# 3. Procesar evaluación con DeepfakeDetector.ai (Posición 1 -> índice 0)
if len(dataset_tuple) > 0:
    print(f"\nIniciando/reanudando evaluación con DeepfakeDetector.ai para la posición 1...\n")

    for idx, item in enumerate(dataset_tuple):
        clasificacion, tipo_fake, file_path = item
        
        # Obtener el valor actual de la posición 1 (índice 0)
        res_detector_actual = respuestas_modelos[idx][0]
        
        # Verificar si la respuesta actual es nula o contiene algún mensaje de error
        es_invalido_o_error = (
            res_detector_actual is None or 
            not isinstance(res_detector_actual, dict) or
            "error" in res_detector_actual or 
            "error_code" in res_detector_actual or
            "data" not in res_detector_actual
        )
        
        # Omitir peticiones si la posición 1 ya tiene datos válidos
        if not es_invalido_o_error:
            print(f"[{idx+1}/{len(dataset_tuple)}] Omitiendo (ya procesado): {os.path.basename(file_path)}")
            continue
            
        print(f"[{idx+1}/{len(dataset_tuple)}] Procesando DeepfakeDetector.ai: {os.path.basename(file_path)} ({tipo_fake})")
        
        try:
            if not os.path.exists(file_path):
                res_detector = {"error": f"El archivo no existe en la ruta: {file_path}"}
            else:
                # PASO 1: Solicitar la URL firmada de subida y el upload_id
                headers_step1 = {"Authorization": f"Bearer {API_KEY}"}
                res_step1 = requests.post(f"{BASE_URL}/uploads/image", headers=headers_step1)
                res_step1.raise_for_status()
                
                data_step1 = res_step1.json()
                upload_id = data_step1["upload_id"]
                upload_url = data_step1["upload_url"]

                # PASO 2: Subir el archivo binario mediante PUT
                with open(file_path, "rb") as f:
                    res_step2 = requests.put(upload_url, data=f)
                    res_step2.raise_for_status()

                # PASO 3: Solicitar la detección enviando el upload_id
                headers_step3 = {
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json"
                }
                payload_step3 = {"upload_id": upload_id}
                
                res_step3 = requests.post(f"{BASE_URL}/detect/image", headers=headers_step3, json=payload_step3)
                
                if res_step3.status_code in [200, 201]:
                    res_detector = res_step3.json()
                else:
                    try:
                        res_detector = {"error_code": res_step3.status_code, "detail": res_step3.json()}
                    except Exception:
                        res_detector = {"error_code": res_step3.status_code, "message": res_step3.text}

        except Exception as e:
            res_detector = {"error": str(e)}
        
        # Guardar o actualizar la respuesta en la POSICIÓN 1 (índice 0)
        respuestas_modelos[idx][0] = res_detector
        
        # Guardado progresivo tras cada petición para asegurar el avance
        with open(filename_json, "w", encoding="utf-8") as out_file:
            json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)

        time.sleep(0.1)

    print(f"\n✓ Proceso finalizado. Respuestas asignadas a la posición 1.")
    print(f"✓ Archivo '{filename_json}' actualizado con éxito.")

else:
    print("❌ No hay datos que procesar.")

✓ Cargadas 1000 imágenes desde 'dataset_tuple.json'
✓ Cargadas 1000 respuestas previas desde 'respuestas_modelos.json'

Iniciando/reanudando evaluación con DeepfakeDetector.ai para la posición 1...

[1/1000] Omitiendo (ya procesado): real_001.jpg
[2/1000] Procesando DeepfakeDetector.ai: real_002.jpg (real)
[3/1000] Procesando DeepfakeDetector.ai: real_003.jpg (real)
[4/1000] Procesando DeepfakeDetector.ai: real_004.jpg (real)
[5/1000] Procesando DeepfakeDetector.ai: real_005.jpg (real)
[6/1000] Procesando DeepfakeDetector.ai: real_006.jpg (real)
[7/1000] Procesando DeepfakeDetector.ai: real_007.jpg (real)
[8/1000] Procesando DeepfakeDetector.ai: real_008.jpg (real)
[9/1000] Procesando DeepfakeDetector.ai: real_009.jpg (real)
[10/1000] Procesando DeepfakeDetector.ai: real_010.jpg (real)
[11/1000] Procesando DeepfakeDetector.ai: real_011.jpg (real)
[12/1000] Procesando DeepfakeDetector.ai: real_012.jpg (real)
[13/1000] Procesando DeepfakeDetector.ai: real_013.jpg (real)
[14/1000] Procesa